In [86]:
import ollama
from ollama import chat
from ollama import ChatResponse
from pydantic import BaseModel
import json
import pandas as pd

In [87]:
df = pd.read_csv("../data/lisa_sheets.csv")

In [88]:
file_path = "../data/train_test_split/test_folders.json"

In [89]:
with open(file_path, "r", encoding="utf-8") as file:
    test_folders = json.load(file)

len(test_folders)

111

In [90]:
df_test = df[df.folder.isin(test_folders)][:30]
df_test.head()

,folder,id,content_raw,rubric
99,IC-005,OIC-005-01-A,{{objectif de connaissance\n|Identifiant=OIC-0...,Définition
100,IC-005,OIC-005-02-A,{{objectif de connaissance\n|Identifiant=OIC-0...,Définition
101,IC-005,OIC-005-03-A,{{objectif de connaissance\n|Identifiant=OIC-0...,Définition
102,IC-005,OIC-005-04-A,{{objectif de connaissance\n|Identifiant=OIC-0...,Définition
103,IC-005,OIC-005-05-A,{{objectif de connaissance\n|Identifiant=OIC-0...,Définition


In [91]:
class MCQQuestion(BaseModel):
    question: str
    option_a: str
    option_b: str
    option_c: str
    option_d: str
    correct_option: str

In [92]:
from ollama import generate

def generate_mcq(content, model_name, temperature):
    prompt = f"""
    À partir du contenu éducatif suivant, générez une question à choix multiple avec quatre options de réponse dont une seule est correcte.
    La question doit évaluer la compréhension des idées principales, et les options doivent être claires, informatives et pertinentes.
    Assurez-vous que les distracteurs (options incorrectes) suivent une interprétation logique mais incorrecte, basée sur des idées reçues ou des incompréhensions courantes du sujet.  
    Les options de réponse doivent être aussi courtes que possible.

    **Contenu éducatif**

    {content}
    """
    
    generate_params = {
        'model': model_name,
        'options': {'temperature': temperature, 'num_ctx': 8192, 'top_p': 1}, 
        'prompt': prompt,
        'format': MCQQuestion.model_json_schema()
    }
    
    # Get a response
    response = generate(**generate_params)
    
    return response['response']

In [77]:
%%time
df_test['generated_questions_0.1'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="llama3.2:1b-instruct-q8_0", temperature=0.1)
)

CPU times: user 120 ms, sys: 3.65 ms, total: 124 ms
Wall time: 1min 18s


In [78]:
%%time
df_test['generated_questions_0.5'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="llama3.2:1b-instruct-q8_0", temperature=0.5)
)

CPU times: user 125 ms, sys: 3.59 ms, total: 129 ms
Wall time: 47.5 s


In [79]:
%%time
df_test['generated_questions_0.7'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="llama3.2:1b-instruct-q8_0", temperature=0.7)
)

CPU times: user 119 ms, sys: 0 ns, total: 119 ms
Wall time: 1min 6s


In [80]:
from pydantic import ValidationError

In [81]:
def validate_mcq(mcq_json):
    try:
        return MCQQuestion.model_validate_json(mcq_json)
    except ValidationError as e:
        print(f"Validation failed: {e}")
        return None
        


In [82]:
df_test["validated_mcq_0.1"] = df_test['generated_questions_0.1'].apply(validate_mcq)

In [83]:
df_test["validated_mcq_0.5"] = df_test['generated_questions_0.5'].apply(validate_mcq)

In [84]:
df_test["validated_mcq_0.7"] = df_test['generated_questions_0.7'].apply(validate_mcq)

In [85]:
df_test.to_csv('llama1b_mcqs.csv', index=False)

In [93]:
from pandas import DataFrame

def flatten_and_export_mcq(df: DataFrame, export_filename: str, mcq_column_name: str):
    result_df = df[['id']].copy()
    
    result_df['question'] = df[mcq_column_name].apply(lambda x: x.question if x else "")
    result_df['option_a'] = df[mcq_column_name].apply(lambda x: x.option_a if x else "")
    result_df['option_b'] = df[mcq_column_name].apply(lambda x: x.option_b if x else "")
    result_df['option_c'] = df[mcq_column_name].apply(lambda x: x.option_c if x else "")
    result_df['option_d'] = df[mcq_column_name].apply(lambda x: x.option_d if x else "")
    result_df['correct_option'] = df[mcq_column_name].apply(lambda x: x.correct_option if x else "")
    
    result_df.to_csv(export_filename, index=False)

In [96]:
flatten_and_export_mcq(df_test, '../data/base_models/llama1b/temp0.1.csv', 'validated_mcq_0.1')
flatten_and_export_mcq(df_test, '../data/base_models/llama1b/temp0.5.csv', 'validated_mcq_0.5')
flatten_and_export_mcq(df_test, '../data/base_models/llama1b/temp0.7.csv', 'validated_mcq_0.7')

KeyError: 'validated_mcq_0.1'

In [ ]:
# One-liner version
df_test[df_test['validated_mcq_0.5'].isna()].index.tolist()

[]

In [ ]:
empty_mcq = MCQQuestion(
    question="",
    option_a="",
    option_b="",
    option_c="",
    option_d="",
    correct_option=""
)

# Set the value at the specified index
df_test.at[3742, 'validated_mcq_0.5'] = empty_mcq

TypeError: object of type 'MCQQuestion' has no len()